# Model Layer Smoke Test Notebook

Interactive version of `scripts/run_model_smoke_test.py`.

Default settings match the smoke script:

1. Load `data/processed/model_inputs/`.
2. Restrict to the first 3 regions as a closed mini-economy.
3. Apply a 30% supply shock to the first 5 productive nodes.
4. Run `IOClimateModel` with the reference strict-min model.
5. Inspect convergence, loss totals, and changed flows.

Run `scripts/build_model_inputs_from_sam.py` first if model inputs are missing or stale.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = next(
        p for p in [PROJECT_ROOT, *PROJECT_ROOT.parents]
        if (p / "src").exists()
    )

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import json
import numpy as np
import pandas as pd

from climate_risk_io.model import input_builder, results
from climate_risk_io.model.io_model import IOClimateModel
from config.paths import MODEL_INPUTS_DIR
from scripts.run_model_smoke_test import subset_inputs

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

print("Project root:", PROJECT_ROOT)
print("Model inputs:", MODEL_INPUTS_DIR)


## 1. Load Model Inputs

The current builder writes `IMP0.npy` for ROW/import rows into productive columns. The propagation model still runs on the productive block (`Z0`, `FD0`, `X0`, `globsec_of`), while `IMP0` is used for accounting diagnostics.


In [ ]:
inputs_full = input_builder.load_model_inputs(MODEL_INPUTS_DIR)
report_path = MODEL_INPUTS_DIR / "model_input_report.json"
report = json.loads(report_path.read_text(encoding="utf-8")) if report_path.exists() else {}

nodes_full = inputs_full["productive_nodes"]
print(f"Productive nodes: {len(nodes_full)}")
print(f"Z0 shape: {inputs_full['Z0'].shape}")
print(f"FD0 shape: {inputs_full['FD0'].shape}")
print(f"X0 shape: {inputs_full['X0'].shape}")
print(f"IMP0 shape: {inputs_full['IMP0'].shape}")
print(f"Regions: {nodes_full['region_code'].nunique()}, sectors: {nodes_full['sector_code'].nunique()}")

if report:
    diag = report["row_vs_column_output"]
    print("\nAccounting diagnostic:")
    print(f"  max relative gap with IMP0 : {diag['max_rel_error']:.6e}")
    print(f"  max relative gap VA only   : {diag['va_only']['max_rel_error']:.6e}")
    print(f"  total import inputs (IMP0) : {diag['total_import_inputs']:,.4f}")

nodes_full.head()


## 2. Smoke-Test Configuration

These defaults are intentionally the same as `scripts/run_model_smoke_test.py`.

Use `USE_FULL = True` only when you are ready for the full dense solve. The subset helper is imported directly from the smoke script so notebook and CLI behavior stay aligned.


In [ ]:
USE_FULL = False
N_REGIONS = 3
SHOCK = 0.30
GAMMA = 0.5
TOP_N = 10

if USE_FULL:
    inputs = inputs_full
    print("Using full productive block.")
else:
    inputs = subset_inputs(inputs_full, N_REGIONS)
    print(f"Subset to first {N_REGIONS} regions.")

nodes = inputs["productive_nodes"]
n = len(nodes)
print(f"Productive nodes in run: {n}")
nodes.head()


## 3. Build Model And Shock Vectors

The smoke script applies an artificial supply shock to the first five productive nodes and no demand shock.


In [ ]:
model = IOClimateModel(
    Z0=inputs["Z0"],
    FD0=inputs["FD0"],
    X0=inputs["X0"],
    sector_group_of=inputs["globsec_of"],
    node_labels=nodes["node_label"].tolist(),
)

sp = np.zeros(n, dtype=float)
sp[: min(5, n)] = SHOCK
sd = np.zeros(n, dtype=float)

print(f"Model built: {model.n} nodes, {model.S_glob} sector groups.")
print(f"Applying supply shock of {SHOCK:.0%} to {min(5, n)} nodes:")
nodes.loc[sp > 0, ["node_id", "region_code", "sector_code", "node_label"]]


## 4. Run The Model

This uses the restored original logic: `Z_new` reallocates flows, the strict global-sector technology cap computes feasible output, and demand is reduced with the reference `FD_implied = X_supply - row_sum(Z_new)` update.


In [ ]:
run_output = model.run(
    sd=sd,
    sp=sp,
    gamma=GAMMA,
    max_iter=50,
    tol=1e-6,
    return_history=True,
)

print("converged :", run_output["converged"])
print("iterations:", run_output["iterations"])
print(f"demand gap: {run_output['demand_update_gap_last']:.6e}")
print("variant   :", run_output["model_variant"])


## 5. Loss Totals

Same KPI summary as the smoke script.


In [ ]:
res = results.summarize_run(
    run_output,
    inputs["Z0"],
    inputs["X0"],
    nodes,
    top_n=TOP_N,
)

print(f"direct_loss   : {res.totals['total_direct_loss']:.4f}")
print(f"indirect_loss : {res.totals['total_indirect_loss']:.4f}")
print(f"total_loss    : {res.totals['total_loss']:.4f}")
print(f"x_pre total   : {res.totals['total_x_pre']:.4f}")
print(f"x_post total  : {res.totals['total_x_post']:.4f}")
print(f"loss rate     : {res.totals['total_loss'] / res.totals['total_x_pre']:.2%}")


## 6. Most Affected Nodes

In [ ]:
per_node = res.per_node_table(nodes)
node_cols = [
    "node_id", "region_code", "sector_code", "x_pre", "x_post",
    "direct_loss", "indirect_loss", "total_loss", "loss_rate",
]
per_node.sort_values("total_loss", ascending=False)[node_cols].head(15)


## 7. Changed Flows

`delta_Z = Z_final - Z0`. Penalised flows shrink the most; favoured flows grow through within-sector reallocation.


In [ ]:
print("Top penalised flows")
res.top_penalized_flows


In [ ]:
print("Top favoured flows")
res.top_favored_flows


## 8. Convergence History

In [ ]:
history = pd.DataFrame({
    "iteration": np.arange(1, len(run_output["X_supply_history"]) + 1),
    "demand_update_gap": run_output["demand_update_gap_history"],
    "fd_total": [float(x.sum()) for x in run_output["FD_post_history"]],
    "x_supply_total": [float(x.sum()) for x in run_output["X_supply_history"]],
})
history.tail(10)


## 9. Sanity Check: Zero Shock Is Lossless

In [ ]:
baseline = model.run(
    sd=np.zeros(n),
    sp=np.zeros(n),
    gamma=GAMMA,
)
max_dev = float(np.max(np.abs(baseline["X_supply_final"] - inputs["X0"])))
print(f"Max deviation from baseline output: {max_dev:.3e}")
assert np.allclose(baseline["X_supply_final"], inputs["X0"], rtol=1e-6, atol=1e-6)
print("OK: zero shock reproduces baseline output.")


## 10. Optional Custom Single-Node Shock

Use this cell when you want a targeted scenario instead of the smoke-test shock. It does not run automatically unless `RUN_CUSTOM = True`.


In [ ]:
RUN_CUSTOM = False
CUSTOM_SHOCK_SIZE = 0.40
CUSTOM_REGION = nodes["region_code"].iloc[0]
CUSTOM_SECTOR = "A01"

if RUN_CUSTOM:
    sp_custom = np.zeros(n, dtype=float)
    sd_custom = np.zeros(n, dtype=float)
    mask = (nodes["region_code"] == CUSTOM_REGION) & (nodes["sector_code"] == CUSTOM_SECTOR)
    hit = np.where(mask.to_numpy())[0]
    if len(hit) == 0:
        raise ValueError(f"No node for {CUSTOM_REGION}/{CUSTOM_SECTOR} in this run.")
    sp_custom[hit] = CUSTOM_SHOCK_SIZE
    custom_output = model.run(
        sd=sd_custom,
        sp=sp_custom,
        gamma=GAMMA,
        max_iter=50,
        tol=1e-6,
    )
    custom_res = results.summarize_run(custom_output, inputs["Z0"], inputs["X0"], nodes, top_n=TOP_N)
    print("converged :", custom_output["converged"])
    print("iterations:", custom_output["iterations"])
    print(f"demand gap: {custom_output['demand_update_gap_last']:.6e}")
    print(custom_res.totals)
else:
    print("Set RUN_CUSTOM = True to run a targeted single-node shock.")
